In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType, StringType, StructType, StructField
from pyspark.sql.functions import regexp_replace, col, regexp_extract, when, date_format, to_date
from pyspark.sql import SparkSession
from kafka import KafkaProducer, KafkaConsumer
from pymongo import MongoClient
import json
from bson import ObjectId
#connect to mongodb
client = MongoClient('mongodb://localhost:27020')
db = client['db_goodread']
collection = db['tb_book']


In [3]:
def json_serializer(document):
    return json.dumps(document, default=lambda x: str(x) if isinstance(x, ObjectId) else x).encode('utf-8')

# Kết nối tới Kafka
producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=json_serializer
)

# Đọc dữ liệu từ MongoDB và gửi tới Kafka
for document in collection.find():
    producer.send('book', document)  # Gửi dữ liệu vào topic "book"

producer.flush()  # Đảm bảo tất cả các bản ghi đã được gửi
producer.close()

In [3]:
# Format data
schema = StructType([
    StructField("author", StringType(), True),
    StructField("bookUrl", StringType(), True),
    StructField("bookname", StringType(), True),
    StructField("describe", StringType(), True),
    StructField("prices", StringType(), True),
    StructField("publish", StringType(), True),
    StructField("rating", StringType(), True),
    StructField("ratingcount", StringType(), True),
    StructField("reviews", StringType(), True),
    StructField("fivestars", StringType(), True),
    StructField("fourstars", StringType(), True),
    StructField("threestars", StringType(), True),
    StructField("twostars", StringType(), True),
    StructField("onestar", StringType(), True),
    StructField("pages", StringType(), True)
])


In [3]:
# Read data from Kafka topic
spark = SparkSession.builder \
        .appName('SparkKafkaToPostgres') \
        .config('spark.jars.packages', "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3," 
                                        "org.postgresql:postgresql:42.5.0") \
        .getOrCreate()

In [15]:
df = spark.readStream.format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("subscribe", "book") \
        .load()

In [7]:
schema = StructType([
    StructField("author", StringType(), True),
    StructField("bookUrl", StringType(), True),
    StructField("bookname", StringType(), True),
    StructField("describe", StringType(), True),
    StructField("prices", StringType(), True),
    StructField("publish", StringType(), True),
    StructField("rating", StringType(), True),
    StructField("ratingcount", StringType(), True),
    StructField("reviews", StringType(), True),
    StructField("fivestars", StringType(), True),
    StructField("fourstars", StringType(), True),
    StructField("threestars", StringType(), True),
    StructField("twostars", StringType(), True),
    StructField("onestar", StringType(), True),
    StructField("pages", StringType(), True)
])

In [9]:
def format_data(df):
    df = df.selectExpr("CAST(value AS STRING)") \
        .selectExpr(f"from_json(value, '{schema.simpleString()}') as jsonData") \
        .select("jsonData.*")
    # Processing string data
    df = df.withColumn("onestar", regexp_replace(col("onestar"), r"[^\d]", "")) \
           .withColumn("twostars", regexp_replace(col("twostars"), r"[^\d]", "")) \
           .withColumn("threestars", regexp_replace(col("threestars"), r"[^\d]", "")) \
           .withColumn("fourstars", regexp_replace(col("fourstars"), r"[^\d]", "")) \
           .withColumn("fivestars", regexp_replace(col("fivestars"), r"[^\d]", "")) \
           .withColumn("pages_n", regexp_extract(col("pages"), r"(\d+)", 1)) \
           .withColumn("cover", regexp_extract(col("pages"), r",\s*(.*)", 1)) \
           .withColumn("prices", when(df["prices"].like("Kindle%"), regexp_extract(col("prices"), r"\$(\d+\.\d{2})", 1)).otherwise(0)) \
           .withColumn("publish", regexp_extract(col("publish"), r"(\w+ \d{1,2}, \d{4})", 1)) \
           .withColumn("publish", to_date(col("publish"), "MMMM d, yyyy")) \
           .withColumn("publish", date_format(col("publish"), "dd/MM/yyyy")) \
           .withColumn("ratingcount", regexp_replace(col("ratingcount"), ",", "")) \
           .withColumn("reviews", regexp_replace(col("reviews"), ",", "")) \
           .drop("pages")
    return df

In [10]:
# Convert data types
def convert(df):
    df = df.withColumn("pages_n", df["pages_n"].cast("int")) \
           .withColumn("prices", df["prices"].cast("float")) \
           .withColumn("onestar", df["onestar"].cast("int")) \
           .withColumn("twostars", df["twostars"].cast("int")) \
           .withColumn("threestars", df["threestars"].cast("int")) \
           .withColumn("fourstars", df["fourstars"].cast("int")) \
           .withColumn("fivestars", df["fivestars"].cast("int")) \
           .withColumn("publish", df["publish"].cast("date")) \
           .withColumn("rating", df["rating"].cast("float")) \
           .withColumn("ratingcount", df["ratingcount"].cast("int")) \
           .withColumn("reviews", df["reviews"].cast("int"))
    return df

In [11]:
df = format_data(df)

In [13]:
df.show()

AnalysisException: Queries with streaming sources must be executed with writeStream.start();
kafka

In [ ]:
if __name__ == "__main__":
    df = format_data(df)
    df = convert(df)
    df.writeStream \
        .format("console") \
        .outputMode("append") \
        .start() \
        .awaitTermination()